In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.model_selection import train_test_split

In [3]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("flight_delay_base")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1786818152302, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786818152302, lifecycle_stage='active', name='flight_delay_base', tags={}, trace_location=None, workspace='default'>

In [4]:
%pwd

'/Users/nicholasstanfield/Desktop/flight-delay/notebooks'

In [5]:
df = pd.read_csv("../data/processed/flight_data_2025.csv")
df.head()

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,AIRLINE
0,1,1,1,3,AUS,ORD,545,830,0.0,165.0,977.0,American Airlines Inc.
1,1,1,7,2,PBI,DFW,620,842,0.0,202.0,1102.0,American Airlines Inc.
2,1,1,26,7,LAS,DEN,920,1219,0.0,119.0,628.0,United Air Lines Inc.
3,1,1,17,5,ICT,ATL,1750,2105,0.0,135.0,782.0,Delta Air Lines Inc.
4,1,1,14,2,CHS,EWR,1930,2129,0.0,119.0,628.0,United Air Lines Inc.


In [6]:
df["ARR_DEL15"] = df["ARR_DEL15"].astype(int)

In [7]:
df["ROUTE"] = df["ORIGIN"] + "-" + df["DEST"]

In [8]:
df["ROUTE"].value_counts()

ROUTE
LAX-SFO    593
SFO-LAX    590
ORD-LGA    582
HNL-OGG    572
LGA-ORD    563
          ... 
IAH-FCA      1
BIS-SFB      1
IAD-STL      1
IAH-ATW      1
SFO-RSW      1
Name: count, Length: 6493, dtype: int64

In [9]:
(df["ROUTE"].value_counts() < 5).sum()

np.int64(898)

In [10]:
counts = df["ROUTE"].value_counts()
for threshold in [2, 5, 10, 20, 50]:
    rare = counts[counts < threshold]
    print(
        threshold,
        "routes:", len(rare),
        "flights:", rare.sum(),
        "% data:", round(rare.sum() / len(df) * 100,2)
    )

2 routes: 272 flights: 272 % data: 0.08
5 routes: 898 flights: 2143 % data: 0.6
10 routes: 1589 flights: 6846 % data: 1.9
20 routes: 2557 flights: 20838 % data: 5.79
50 routes: 4116 flights: 72505 % data: 20.14


* There are some routes which only appear once or twice in the whole dataset meaning there is too little data to capture any meaningful patterns specific to that route
* So we drop all routes with 10 or less observations (~7000 rows)

In [11]:
valid_routes = counts[counts >= 10].index
df = df[df["ROUTE"].isin(valid_routes)]

In [12]:
X = df.drop("ARR_DEL15",axis=1)
y = df["ARR_DEL15"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [14]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((282523, 12), (70631, 12), (282523,), (70631,))

In [15]:
categorical_columns = X_train.select_dtypes(include=["object"]).columns.to_list()

numeric_columns = X_train.select_dtypes(include=["int","float"]).columns.to_list()

In [16]:
categorical_columns, '-----------', numeric_columns

(['ORIGIN', 'DEST', 'AIRLINE', 'ROUTE'],
 '-----------',
 ['QUARTER',
  'MONTH',
  'DAY_OF_MONTH',
  'DAY_OF_WEEK',
  'CRS_DEP_TIME',
  'CRS_ARR_TIME',
  'CRS_ELAPSED_TIME',
  'DISTANCE'])

In [17]:
X["ORIGIN"].nunique(),X["DEST"].nunique(),X["AIRLINE"].nunique()

(335, 332, 14)

In [18]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


cat_pipeline = Pipeline([("one_hot_encoder", OneHotEncoder(handle_unknown='ignore'))])
num_pipeline = Pipeline([("scale", StandardScaler())])


preprocessing = ColumnTransformer([
    ("num", num_pipeline, numeric_columns),
    ("cat", cat_pipeline, categorical_columns),
])

In [19]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

(ARR_DEL15
 0    0.77828
 1    0.22172
 Name: proportion, dtype: float64,
 ARR_DEL15
 0    0.778284
 1    0.221716
 Name: proportion, dtype: float64)

In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# for the scoring add a zero_division argument to avoid warnings on model fit
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

In [24]:
import mlflow.sklearn

dummy_classifier = make_pipeline(
    preprocessing,
    DummyClassifier(strategy="prior")
)

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="dummy_classifier_adjusted_metrics"):

    cv_results = cross_validate(
        dummy_classifier,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    metrics = {
        "cv_accuracy_mean": round(cv_results["test_accuracy"].mean(), 2),
        "cv_precision_mean": round(cv_results["test_precision"].mean(), 2),
        "cv_recall_mean": round(cv_results["test_recall"].mean(), 2),
        "cv_f1_mean": round(cv_results["test_f1"].mean(), 2),
    }

    mlflow.log_metrics(metrics)

metrics

🏃 View run dummy_classifier_adjusted_metrics at: http://localhost:5001/#/experiments/1/runs/5980653630ce40ebaa63badc18ef41ae
🧪 View experiment at: http://localhost:5001/#/experiments/1


{'cv_accuracy_mean': np.float64(0.7782799984509946),
 'cv_precision_mean': np.float64(0.0),
 'cv_recall_mean': np.float64(0.0),
 'cv_f1_mean': np.float64(0.0)}

In [25]:
from sklearn.linear_model import LogisticRegression

logistic_regression = make_pipeline(
    preprocessing,
    LogisticRegression(
        solver="saga",
        max_iter=10_000,
        random_state=42
    )
)

with mlflow.start_run(run_name="logistic_regression_increased_iters"):

    cv_results = cross_validate(
        logistic_regression,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    metrics = {
        "cv_accuracy_mean": round(cv_results["test_accuracy"].mean(), 2),
        "cv_precision_mean": round(cv_results["test_precision"].mean(), 2),
        "cv_recall_mean": round(cv_results["test_recall"].mean(), 2),
        "cv_f1_mean": round(cv_results["test_f1"].mean(), 2),
    }


    mlflow.log_metrics(metrics)

metrics

🏃 View run logistic_regression_increased_iters at: http://localhost:5001/#/experiments/1/runs/f46b98561eaf4c199b733dfaaa99484d
🧪 View experiment at: http://localhost:5001/#/experiments/1


{'cv_accuracy_mean': np.float64(0.7765420853062197),
 'cv_precision_mean': np.float64(0.4242624564874637),
 'cv_recall_mean': np.float64(0.021934498935649367),
 'cv_f1_mean': np.float64(0.04171029315714477),
 'cv_accuracy_std': np.float64(0.00029274193307516104),
 'cv_precision_std': np.float64(0.012555406062480286),
 'cv_recall_std': np.float64(0.0008947052754811472),
 'cv_f1_std': np.float64(0.00165487337572745)}

In [26]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = make_pipeline(
    preprocessing,
    DecisionTreeClassifier(random_state=42)
)

with mlflow.start_run(run_name="decision_tree"):

    cv_results = cross_validate(
        decision_tree,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    metrics = {
    "cv_accuracy_mean": round(cv_results["test_accuracy"].mean(), 2),
    "cv_precision_mean": round(cv_results["test_precision"].mean(), 2),
    "cv_recall_mean": round(cv_results["test_recall"].mean(), 2),
    "cv_f1_mean": round(cv_results["test_f1"].mean(), 2),
    }

    mlflow.log_metrics(metrics)

metrics

🏃 View run decision_tree at: http://localhost:5001/#/experiments/1/runs/54b697fcaefa42bab6ba9357cf74543a
🧪 View experiment at: http://localhost:5001/#/experiments/1


{'cv_accuracy_mean': np.float64(0.72),
 'cv_precision_mean': np.float64(0.34),
 'cv_recall_mean': np.float64(0.29),
 'cv_f1_mean': np.float64(0.31)}

In [27]:
from sklearn.ensemble import RandomForestClassifier

random_forest = make_pipeline(
    preprocessing,
    RandomForestClassifier(
        random_state=42,
        n_jobs=1
    )
)

with mlflow.start_run(run_name="random_forest"):

    cv_results = cross_validate(
        random_forest,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    metrics = {
        "cv_accuracy_mean": round(cv_results["test_accuracy"].mean(), 2),
        "cv_precision_mean": round(cv_results["test_precision"].mean(), 2),
        "cv_recall_mean": round(cv_results["test_recall"].mean(), 2),
        "cv_f1_mean": round(cv_results["test_f1"].mean(), 2),
    }

    mlflow.log_metrics(metrics)

metrics

🏃 View run random_forest at: http://localhost:5001/#/experiments/1/runs/f57da835824742e0bce3b88398a61951
🧪 View experiment at: http://localhost:5001/#/experiments/1


{'cv_accuracy_mean': np.float64(0.77),
 'cv_precision_mean': np.float64(0.44),
 'cv_recall_mean': np.float64(0.11),
 'cv_f1_mean': np.float64(0.17)}

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier


cat_pipeline = Pipeline([("one_hot_encoder", OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
num_pipeline = Pipeline([("scale", StandardScaler())])


preprocessing = ColumnTransformer([
    ("num", num_pipeline, numeric_columns),
    ("cat", cat_pipeline, categorical_columns),
])


hist_gradient_boosting = make_pipeline(
    preprocessing,
    HistGradientBoostingClassifier(
        random_state=42
    )
)

with mlflow.start_run(run_name="hist_gradient_boosting"):

    cv_results = cross_validate(
        hist_gradient_boosting,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    metrics = {
        "cv_accuracy_mean": round(cv_results["test_accuracy"].mean(), 2),
        "cv_precision_mean": round(cv_results["test_precision"].mean(), 2),
        "cv_recall_mean": round(cv_results["test_recall"].mean(), 2),
        "cv_f1_mean": round(cv_results["test_f1"].mean(), 2),
    }

    mlflow.log_metrics(metrics)

metrics

🏃 View run hist_gradient_boosting at: http://localhost:5001/#/experiments/1/runs/b5974e7092c9466dab741505c6cb2a05
🧪 View experiment at: http://localhost:5001/#/experiments/1


{'cv_accuracy_mean': np.float64(0.78),
 'cv_precision_mean': np.float64(0.65),
 'cv_recall_mean': np.float64(0.06),
 'cv_f1_mean': np.float64(0.1)}

In [ ]:
from xgboost import XGBClassifier

xgboost_classifier = make_pipeline(
    preprocessing,
    XGBClassifier(
        random_state=42,
        n_jobs=1,
        eval_metric="logloss"
    )
)

with mlflow.start_run(run_name="xgboost_classifier"):

    cv_results = cross_validate(
        xgboost_classifier,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    metrics = {
        "cv_accuracy_mean": round(cv_results["test_accuracy"].mean(), 2),
        "cv_precision_mean": round(cv_results["test_precision"].mean(), 2),
        "cv_recall_mean": round(cv_results["test_recall"].mean(), 2),
        "cv_f1_mean": round(cv_results["test_f1"].mean(), 2),
    }

    mlflow.log_metrics(metrics)

metrics

## TODO

* Use MLFlow for each experiment 
* Split data into X, y (DONE)
* Split the data into train-test (use CV so no need for a validation set) (DONE)
* One hot encode all categorical columns (DONE)
* Explore some feature engineering such as route create, speed column (distance over time) (IN PROGRESS)
* Try the following models:
    * Logistic Regression (IN PROGRESS)
    * SVM
    * Decision Trees
    * Random Forest
    * Gradient Boosting
    * XGBoost (and similiar, e.g. LightGBM, CatBoost)
    * Neural Networks (through sklearn)
* Export final model into a model folder (DO NOT DO THIS IN THIS NOTEBOOK TAKE THE BEST MODEL AND WRITE THE CODE IN A model_pipeline.py in src
    * Not just the pipeline but the preprocessing as well so the prediction can be easily fed in